In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tiktoken
import torch
import torch.nn as nn
from torch.nn import functional as F
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sakshisharma221/shakespeare-text/input.txt


In [2]:
print(torch.cuda.get_device_name(0))  # tells you which GPU you got"

Tesla T4


In [3]:
with open("/kaggle/input/datasets/sakshisharma221/shakespeare-text/input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [4]:
len(text)  #1M

1115394

In [5]:
unique_chars = sorted(list(set(text)))
print("".join(unique_chars))

stoi = {ch: i for i, ch in enumerate(unique_chars)}
itos = {i: ch for i, ch in enumerate(unique_chars)}

# stoi = {}
# itos = {}
# for i, c in enumerate(unique_chars):
#     stoi[c] = i
#     itos[i] = c


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [6]:
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda s: "".join([itos[i] for i in s])

In [7]:
data = torch.tensor(encode(text), dtype=torch.long) 

In [8]:
data.shape

torch.Size([1115394])

In [9]:
data.dtype

torch.int64

In [10]:
data[:1000]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
        47, 59, 57,  1, 47, 57,  1, 41, 

In [11]:
print(decode(data[:1000].tolist()))

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [12]:
#hyperparameters

vocab_size = len(unique_chars) #65
batch_size = 32 # independent sequence we will process in parallel
block_size = 64 # maximum context length for predictions
n_embd = 128
num_heads = 4
head_size = n_embd // num_heads # key, query, value vector size # 32
eval_iter = 50
max_iter = 10000
learning_rate = 3e-4
n_layers = 4 # no. of blocks
p = 0.2 # dropout rate
device = 'cuda' if torch.cuda.is_available() else 'cpu'

#splitting the data into training and validation sets
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [13]:
torch.manual_seed(1337)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, )) #start index
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device) 
    return x, y

xb, yb = get_batch("train")
print(xb)
print(yb)

tensor([[50, 43,  8,  ..., 53, 51, 43],
        [ 1, 58, 46,  ..., 33, 25, 14],
        [42,  1, 61,  ..., 51,  1, 52],
        ...,
        [46, 47, 52,  ...,  1, 58, 46],
        [51, 54, 43,  ...,  1, 60, 43],
        [10,  1, 58,  ..., 59, 57,  6]], device='cuda:0')
tensor([[43,  8,  1,  ..., 51, 43, 52],
        [58, 46, 43,  ..., 25, 14, 17],
        [ 1, 61, 47,  ...,  1, 52, 53],
        ...,
        [47, 52, 45,  ..., 58, 46, 43],
        [54, 43, 39,  ..., 60, 43, 52],
        [ 1, 58, 46,  ..., 57,  6,  0]], device='cuda:0')


## Decoder

In [14]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(p)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        
    def forward(self, x):
        B, T, C = x.shape # C would be 32
        k = self.key(x) # b * t * 16 # what do i offer
        q = self.query(x) # b * t * 16 # what i am looking for

        # compute attention scores
        wei = q @ k.transpose(-2, -1) * head_size**-0.5 # b * t * t 
        # for only considering previous and current word only
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) 
        wei = F.softmax(wei, dim=-1)

        wei = self.dropout(wei) #randomly turning off some past tokens
        
        v = self.value(x) # what will i contribute
        out = wei @ v # b* t * 16
        return out

In [15]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(p)
        # list of layers

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # this will just concatenate all heads
        out = self.proj(out) # and this will blend them all
        out = self.dropout(out)
        return out 
        # (b, t, n_embd)

In [16]:
# adding a layer for non linearity
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd * 4), # expanding to *4 gives the model more room to do rich computation that is it can learn more useful patterns
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(p)
        )

    def forward(self, x):
        out = self.net(x)
        return out

In [17]:
class Block(nn.Module):
    def __init__(self, n_embd, num_heads):
        super().__init__()
        # n-embd = num_heads * head_size
        head_size = n_embd // num_heads
        self.sa_heads = MultiHeadAttention(num_heads, head_size)
        self.ffwl = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        
    def forward(self, x):
        # pre norm
        x = x + self.sa_heads(self.ln1(x))
        x = x + self.ffwl(self.ln2(x))
        return x

In [18]:
@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            xb, yb = get_batch(split)
            logits, loss = m(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out
    

In [19]:
torch.manual_seed(1337)

class Decoder(nn.Module):

    def __init__(self):
        super().__init__()
        # nn.embedding(num_tokens, vector_size)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # lookup table: for each token, stores a vector of size vocab_size
        # these vectors are learned during training

        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.blocks = nn.Sequential(
            # * is used for unpacking
            *[Block(n_embd, num_heads) for _ in range(n_layers)],
            nn.LayerNorm(n_embd)
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            # initialize linear layers
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
    
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        
        
    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_emb = self.token_embedding_table(idx) # B * T * C
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # T * C
        x = token_emb + pos_emb # B * T * n_embd
        x = self.blocks(x)
        # linear layer
        logits = self.lm_head(x) # B * T * vocab size #
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape  # c here is vocab size
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)    
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
            
    # max new tokens - how many new tokens to generate
    # generates many tokens in a loop, one at a time, each time feeding the previous output back in
    def generate(self, idx,  max_new_tokens):

        

        # [54, 1, 48] 
        # ->
        # [[vector], [vector], [vector]]
        # -> last one [[vector]]
        # -> [[probs vector]]
        # -> [get max idx in that prob]
        # -> append it to idxs
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond) # calling forward
            logits = logits[:, -1, :] # last
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=-1)
        return idx
        
    
m = Decoder()
m = m.to(device)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss) # first loss is in a ideal case expected to be 4.17  because everyone should get -log(1/65)

torch.Size([2048, 65])
tensor(4.2163, device='cuda:0', grad_fn=<NllLossBackward0>)


In [20]:
idx = torch.zeros((1, 1), dtype=torch.long, device=device) # [[0]]
generated = m.generate(idx, max_new_tokens=1000)

print(decode(generated[0].tolist()))


MBP$ZGFDtESHx;wbERFmr?yto-RihwsGLot3;TL;d
wfbxmndBj!jV.mBM DfuvUaMMMrlfeJnL
shzvKFrP
GcUxghEj;bEGx,jGlECkZfNvLelcSUcmR&
nYSPxhonhnHqrz&kCShIPealaInOeuwHWkEOEBSMkGZfJK
Imy:s!BxnFrkLbkRCb.TriirGqn-,KzZ!wIkP;.WeY.?gvIOKUELUDJhKogm,e
.gkEI,MCkAgOEflJLi!qIbGYN:&pVchLYJa3zPBCKIuQb.JjfX;HYOqk:qxSWypxhFHIcRe Asre,x,u-OMxiSRzmFJgcvIdIGJ?WzbILinx;ZxZV
SwIH$Cs NvVdEH-m$$nQaYfvopw'JJMezw;jh,bUBzILGBeALhQrU-eb!kIY
XgiuwGmrN'CWsjaiyfS..TYPqLrN.Ir-sUMomsAtLnFTJP3!lRKAr3cNQvNz
wnnB&uU
;B-3mis T'rOI?bhoK&
R'Y.AEVuGu 3nIvf$INN:g&HJ.ueFNHwkYCmtxVS$QrhVSCc
Hhkl!;pXXZC3Si3rDZJjhpp-xsRSh;3vi.sIhPHQMS?-myKrWvc,DHB.RkENtcEKnMuQs
HLIid,XWkEaVNpYcIBEndcon-HJH.bsteyQfvbB;iNKYkCAtvRKw,MK!plD.aVe A.QrKsj'fxDnUSn.Y:zT-NoCEJX;wR3ZgOAopf$,.YypBH?.jm&I,cLsutxVRhdOV; p?Uw
tNMrIj z;QlyK!D.';knHSGah&3hpGNEiLtB!faPn
UzmopxqBKdxmXytY!e-U3NBR.u$uuoim-KBEx lqyp,umOaAvAsfqJx,zK:hhq'vWoxZTVBhSTxrzGjJAQC'V.U?efcB-ISIx-
VjjOT$C..L:vvsKwgMMiuRyXd!XL'BdGkLH3l:accatkljDwBZlmE
wvLf-.HsXRmkPzhqWTTd!,R.?SgAhcLvlPKkgk!xzArgxP!ImmmS3E-

In [21]:
# training 

optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)
for epoch in range(max_iter):
    optimizer.zero_grad() # clearing old gradients
    xb, yb = get_batch('train')  # fresh batch every step
    logits, loss = m(xb, yb) # forward pass
    loss.backward() # backward pass
    optimizer.step() # updating weigths
    
    if epoch % 500 == 0 or epoch == max_iter-1:
        losses = estimate_loss()
        print(f"step {epoch}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

step 0: train loss 3.8918, val loss 3.9026
step 500: train loss 2.1864, val loss 2.2097
step 1000: train loss 1.8749, val loss 1.9840
step 1500: train loss 1.7450, val loss 1.8919
step 2000: train loss 1.6511, val loss 1.8100
step 2500: train loss 1.6072, val loss 1.7758
step 3000: train loss 1.5855, val loss 1.7684
step 3500: train loss 1.5446, val loss 1.7349
step 4000: train loss 1.5259, val loss 1.7174
step 4500: train loss 1.4988, val loss 1.7098
step 5000: train loss 1.4984, val loss 1.6785
step 5500: train loss 1.4828, val loss 1.6750
step 6000: train loss 1.4617, val loss 1.6631
step 6500: train loss 1.4478, val loss 1.6584
step 7000: train loss 1.4445, val loss 1.6504
step 7500: train loss 1.4352, val loss 1.6515
step 8000: train loss 1.4255, val loss 1.6333
step 8500: train loss 1.4192, val loss 1.6547
step 9000: train loss 1.4052, val loss 1.6176
step 9500: train loss 1.4069, val loss 1.6275
step 9999: train loss 1.3891, val loss 1.6144


In [22]:
idx = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = m.generate(idx, max_new_tokens=1000)
print(decode(generated[0].tolist()))


I wouly fell smell mes, and Mantuanle,
Are but emback consironar full o' thy brother
As thinds too blest agotly, so more in and back o'
wife, to her
your very lie strate;
Gesting I nantime I.

MARCIUS:
I this lord.

ISABELLA:
No, so soon, no noble like my eye.

First CitizellincA3
And lield Icannot all are partal god
To my forth: 'tis she my loves or thee.

MAMILIUS:
See, seek, my lies made op to--light.

LEONTES:
Perdick thee this appoling and histerning gatents?

COMENENIUSARRELE:
But our loves else and more.

SICILIS:
I joy mercius, as to God, my to's reque.

PHENRY ANNIELLO:
This basts come wards. Call wiself you will, and
Even in thelf rage; that o she's here daughter's partlanel's gods,
I shall not to on her, and this to the presut
Says to that meneset here what was a patiently
with obstor unto refull'd! beiside--ton, death
I thought Evy hear is man we shildly take
Hirt enwret be; and in takes mily boe. Look you:
Thou empetr an yet speak a batter Duke,
No, whom do I remain, for 

In [23]:
for name, param in m.named_parameters():
    print(f"{name}: {param.numel()}")
    
print(f"\nTotal: {sum(p.numel() for p in m.parameters()):,}")

token_embedding_table.weight: 8320
position_embedding_table.weight: 8192
lm_head.weight: 8320
lm_head.bias: 65
blocks.0.sa_heads.heads.0.key.weight: 4096
blocks.0.sa_heads.heads.0.query.weight: 4096
blocks.0.sa_heads.heads.0.value.weight: 4096
blocks.0.sa_heads.heads.1.key.weight: 4096
blocks.0.sa_heads.heads.1.query.weight: 4096
blocks.0.sa_heads.heads.1.value.weight: 4096
blocks.0.sa_heads.heads.2.key.weight: 4096
blocks.0.sa_heads.heads.2.query.weight: 4096
blocks.0.sa_heads.heads.2.value.weight: 4096
blocks.0.sa_heads.heads.3.key.weight: 4096
blocks.0.sa_heads.heads.3.query.weight: 4096
blocks.0.sa_heads.heads.3.value.weight: 4096
blocks.0.sa_heads.proj.weight: 16384
blocks.0.sa_heads.proj.bias: 128
blocks.0.ffwl.net.0.weight: 65536
blocks.0.ffwl.net.0.bias: 512
blocks.0.ffwl.net.2.weight: 65536
blocks.0.ffwl.net.2.bias: 128
blocks.0.ln1.weight: 128
blocks.0.ln1.bias: 128
blocks.0.ln2.weight: 128
blocks.0.ln2.bias: 128
blocks.1.sa_heads.heads.0.key.weight: 4096
blocks.1.sa_heads.he